In [1]:
import pdfplumber
import re
import json
from pathlib import Path


In [7]:
# Source - https://stackoverflow.com/a/78674031
# Posted by Wayne, modified by community. See post 'Timeline' for change history
# Retrieved 2026-04-01, License - CC BY-SA 4.0
# Helper function to display multiline strings in Jupyter notebooks

from IPython.core.formatters import BaseFormatter
from IPython.display import display, HTML

class MultilineStringFormatter(BaseFormatter):
    def __call__(self, obj):
        if isinstance(obj, str) and '\n' in obj:
            return f'<pre>{obj}</pre>'
        return None

# Register the custom formatter
ip = get_ipython()
ip.display_formatter.formatters['text/html'].for_type(str, MultilineStringFormatter())


In [ ]:


def extraer_con_columnas(ruta_pdf):
    # Definiendo expresiones regulares para identificar si una línea tiene acordes o si es el título de una canción
    regex_acordes = re.compile(r'^(\s*[A-G][#b]?[-m7M]?(dim|aug|maj)?(\s+|$))+')
    regex_inicio_cancion = re.compile(r'^([A-Z]-\d+\.)\s+(.*)')
    
    # Inicializando objetos.
    canciones = []
    cancion_actual = None

    with pdfplumber.open(ruta_pdf) as pdf:
        for pagina in pdf.pages:
            # Obtener dimensiones de la página
            ancho = pagina.width
            alto = pagina.height
            
            # Definir las cajas (bounding boxes) para columna izquierda y derecha
            # Formato: (x0, y0, x1, y1)
            columna_izq = (0, 0, ancho / 2, alto)
            columna_der = (ancho / 2, 0, ancho, alto)
            
            for bbox in [columna_izq, columna_der]:
                # "Recortamos" la página virtualmente
                seccion = pagina.within_bbox(bbox)
                texto_seccion = seccion.extract_text()
                
                if not texto_seccion:
                    continue
                
                lineas = texto_seccion.split('\n')
                
                for linea in lineas:
                    linea = linea.strip()
                    if not linea or "Para ti es mi música" in linea:
                        continue

                    # Lógica de detección de canciones
                    match_inicio = regex_inicio_cancion.match(linea)
                    if match_inicio:
                        if cancion_actual:
                            canciones.append(cancion_actual)
                        
                        cancion_actual = {
                            "id": match_inicio.group(1).replace('.', ''),
                            "titulo": match_inicio.group(2),
                            "letra": [],
                            "acordes": []
                        }
                        continue

                    if cancion_actual:
                        if regex_acordes.match(linea):
                            acordes = re.findall(r'[A-G][#b]?[-m7M]?(?:dim|aug|maj)?', linea)
                            cancion_actual["acordes"].extend(acordes)
                        elif not linea.isdigit():
                            cancion_actual["letra"].append(linea)

        if cancion_actual:
            canciones.append(cancion_actual)

    # Limpieza final
    for c in canciones:
        c["letra_limpia"] = " ".join(c["letra"])
        c["acordes"] = sorted(list(set(c["acordes"])))

    return canciones

# Uso en el notebook:
# ruta = Path("documentos") / "Para tí EMM (GT).pdf"
# if ruta.exists():
#     resultados = extraer_con_columnas(ruta)
#     print(f"Se extrajeron {len(resultados)} canciones correctamente.")


# 1. Define la ruta de la carpeta y el archivo
carpeta_datos = Path("pdf-files")
nombre_archivo = "libro1.pdf"

# 2. Crea la ruta completa al archivo
ruta_pdf = carpeta_datos / nombre_archivo

# 3. Verifica si el archivo existe antes de abrirlo
if ruta_pdf.exists():
    print(f"Archivo encontrado en: {ruta_pdf}")
    with pdfplumber.open(ruta_pdf) as pdf:
        # Aquí va tu lógica de extracción
        primera_pagina = pdf.pages[0].extract_text()
        print(primera_pagina[:100])
else:
    print("Error: No se encontró el archivo. Revisa el nombre o la carpeta.")

# Ejecución
data = extraer_con_columnas(ruta_pdf)

# Guardar resultado para el RAG
# COMENTANDO PARA EVITAR SOBREESCRIBIR EL ARCHIVO
#with open('cancionero_limpio.json', 'w', encoding='utf-8') as f:
#      json.dump(data, f, ensure_ascii=False, indent=2)

# TODO: Actualmente el indice se carga con ids duplicados, se debe agregar lógica para que no se lea el índice al final.

Archivo encontrado en: pdf-files/libro1.pdf
" P a r a t i e s
m i m ú s i c a
S e ñ o r "
PARA CELEBRACIONES
EUCARÍSTICAS
LIBRO DE CANTOS
CON AC


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No

In [2]:
# Siguiente paso, teniendo el resultado para hacerlo RAG
import chromadb
from chromadb.utils import embedding_functions

# Replace 'archivo.json' with your actual path
with open('cancionero_limpio.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 1. Configurar el cliente y la función de embeddings
client = chromadb.Client()
# Definir la función de embedding (esto se descarga la primera vez)
model_name = "paraphrase-multilingual-MiniLM-L12-v2"
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=model_name)
collection = client.get_or_create_collection(name="cancionero", embedding_function=embedding_fn)

# Crear o conectar a la base de datos (en memoria para este ejemplo)
client = chromadb.Client()
collection = client.get_or_create_collection(
    name="cancionero", 
    embedding_function=embedding_fn
)

# 2. Preparar los datos del JSON
ids = [c["id"] for c in data]
documents = [c["letra_limpia"] for c in data]
metadatos = [{"titulo": c["titulo"], "acordes": str(c["acordes"])} for c in data]

# 3. Indexar
collection.add(
    documents=documents,
    metadatas=metadatos,
    ids=ids
)

# 4. Probar una búsqueda
results = collection.query(
    query_texts=["cantos para la comunión que hablen de pan y vino"],
    n_results=2
)

#print(results["metadatas"])

/Users/rafaelderas/Library/CloudStorage/OneDrive-Personal/Proyectos Dev - Data Sci/cancionero-catolico/cantos-catolicos-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6968.27it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# Si se necesita eliminar la colección para volver a indexar:
# client.delete_collection(name="cancionero")

# 4. Probar una búsqueda
results = collection.query(
    query_texts=["cantos para la comunión que hablen de pan y vino"],
    n_results=2
)
print(results["metadatas"])

[[{'acordes': "['A', 'B7', 'E']", 'titulo': 'TE PRESENTAMOS EL VINO Y EL PAN'}, {'titulo': 'ALEGRES TOMAMOS EL VINO Y EL', 'acordes': "['A', 'C#-', 'D', 'E', 'E7']"}]]


In [4]:

#Función de búsqueda
def buscar_canciones(query, n_resultados=3):
    # La magia ocurre aquí: query_texts se convierte a vector y busca cercanía
    results = collection.query(
        query_texts=[query],
        n_results=n_resultados
    )
    
    # Formatear la salida para el LLM o para el usuario
    salida = []
    for i in range(len(results['ids'][0])):
        item = {
            "id": results['ids'][0][i],
            "titulo": results['metadatas'][0][i]['titulo'],
            "letra": results['documents'][0][i],
            "acordes": results['metadatas'][0][i]['acordes'],
            "distancia": results['distances'][0][i] # Menor distancia = más similar
        }
        salida.append(item)
    
    return salida

# Ejemplo de uso:
busqueda = buscar_canciones("cantos alegres para el momento de la paz", n_resultados=3)
for res in busqueda:
    print(f"Encontrada: {res['titulo']} (ID: {res['id']})")

Encontrada: PAZ EN LA TIERRA (ID: F-40)
Encontrada: LA PAZ ESTÉ CON NOSOTROS (ID: F-39)
Encontrada: CONGRATULATIONS (ID: R-16)


In [ ]:
import os
from groq import Groq 

client_llm = Groq(api_key="<<API_KEY_GOES_HERE>>")

PROMPT_SISTEMA = """
Eres un asistente experto en música litúrgica. Tu tarea es ayudar a elegir cantos basados EXCLUSIVAMENTE en el contexto proporcionado.
Si el usuario pregunta por acordes, dáselos. Si no, solo entrega la letra.
Si ninguna canción en el contexto coincide con lo que pide el usuario, indícalo cortésmente.

CONTEXTO DE CANCIONES:
{contexto}
"""

def generar_respuesta(pregunta_usuario):
    # 1. Recuperar datos de tu base vectorial (Paso 4)
    canciones_encontradas = buscar_canciones(pregunta_usuario, n_resultados=2)
    
    # 2. Formatear el contexto para el LLM
    contexto_texto = ""
    for c in canciones_encontradas:
        contexto_texto += f"\nID: {c['id']} - Título: {c['titulo']}\n"
        contexto_texto += f"Acordes: {c['acordes']}\n"
        contexto_texto += f"Letra: {c['letra']}\n"
        contexto_texto += "-"*20

    # 3. Llamada al modelo barato (Llama 3 8b es muy económico o gratis en Groq)
    response = client_llm.chat.completions.create(
        #model="llama3-8b-8192",
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": PROMPT_SISTEMA.format(contexto=contexto_texto)},
            {"role": "user", "content": pregunta_usuario}
        ],
        temperature=0.2 # Temperatura baja para que no invente letras
    )
    
    return response.choices[0].message.content

In [8]:

respuesta = generar_respuesta(PROMPT_SISTEMA.format(contexto="cantos que hablen del perdón y la misericordia"))

display(respuesta)

'Basado en el contexto proporcionado, te recomiendo la canción "HAZME UN INSTRUMENTO DE TU" (ID: I-24). La letra de esta canción habla sobre el perdón y la misericordia, y cómo podemos ser instrumentos de la paz de Dios.\n\nLa letra es la siguiente:\nPAZ Hazme un instrumento de tu paz, donde haya odio ponga yo tu amor, donde haya injuria tu perdón Señor, donde haya duda fe en Ti. Maestro ayúdame a nunca buscar, querer ser consolado como consolar, ser entendido como entender, ser amado como yo amar. Hazme un instrumento de tu paz que lleve tu esperanza por doquier donde haya oscuridad ponga tu luz, donde haya pena tu gozo Señor. Hazme un instrumento de tu paz, es perdonando que nos das perdón, es dando a otros que tú te nos das, muriendo es que volvemos a nacer. FINAL Hazme un instrumento de tu paz.\n\nEspero que esta canción sea de ayuda para ti. Si necesitas algo más, no dudes en preguntar.'